# LDTF-BERT on AG News — Colab

Label-Directed Token and Depth Fusion over BERT layers.

**Before you start:** set **Runtime -> Change runtime type -> GPU** (T4 is enough).

This notebook trains and validates only. The official test split stays sealed
until the final locked evaluation at the end, which is opt-in and logged.


## 1. Check the GPU

In [ ]:
!nvidia-smi

import torch
print('torch', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
    print('bf16 supported:', torch.cuda.is_bf16_supported())


## 2. Get the code and data

Pick **one** of the two options below.

**Option A — Google Drive (recommended, no git needed).** On your own
machine, from the project root, build a single archive:

```bash
tar --exclude='__pycache__' --exclude='*.pyc' --exclude='outputs' \
    --exclude='src/data' -czf ldtf_bert.tar.gz \
    src experiments scripts tests docs requirements.txt README.md data/processed
```

That is roughly 100 MB (0.6 MB of code plus the three parquet splits).
Upload `ldtf_bert.tar.gz` to your Drive, then set `USE_DRIVE = True` below.

**Option B — git clone.** Only works if the repository is pushed somewhere
Colab can reach, and the parquet files must be included or fetched separately.

Either way, `PROJECT` must end up being the directory that contains `src/`.

In [ ]:
import os, sys
from pathlib import Path

USE_DRIVE = True                     # False to clone from git instead
ARCHIVE   = '/content/drive/MyDrive/ldtf_bert.tar.gz'
REPO_URL  = 'https://github.com/USER/HocSau_LDTF_BERT.git'
PROJECT   = Path('/content/HocSau_LDTF_BERT')

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    assert Path(ARCHIVE).exists(), f'archive not found at {ARCHIVE}'
    PROJECT.mkdir(parents=True, exist_ok=True)
    !tar -xzf $ARCHIVE -C $PROJECT
else:
    if not PROJECT.exists():
        !git clone $REPO_URL $PROJECT

assert (PROJECT / 'src').is_dir(), f'src/ not found under {PROJECT}'
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
print('working directory:', Path.cwd())


## 3. Install dependencies

In [ ]:
!pip install -q -r requirements.txt
print('dependencies installed')


## 4. Check the data

Expected under `data/processed/`: `research_train.parquet`,
`research_validation.parquet`, `research_test.parquet`.
The test file is present but sealed; nothing below reads it.

In [ ]:
from src import config
from src.dataset import load_split

for name, path in [('train', config.PROCESSED_TRAIN), ('validation', config.PROCESSED_VAL)]:
    frame = load_split(path)
    counts = frame[config.LABEL_COLUMN].value_counts().sort_index().tolist()
    print(f'{name:<11} {len(frame):>7,} rows   per-class {counts}')

print('\nofficial test split:', config.PROCESSED_TEST.name,
      '(sealed — see src/guard.py)')


## 5. Smoke test

Drives all 22 configurations through forward, backward, optimizer coverage,
gradient audit, checkpointing, resume and evaluation on a tiny synthetic model.
Takes about a minute and catches an environment problem before a long run.

In [ ]:
!python -m scripts.smoke_test 2>&1 | tail -n 25


## 6. Quick sanity run

A short run on a small subset to confirm the live display works and loss moves.
These are **debug numbers, not results** — the summary records
`is_debug_subset: true`.

In [ ]:
!python -m experiments.run_experiment --run A0 --limit-train-rows 2000 --epochs 1 --batch-size 32


## 7. Train one configuration

The live display is YOLO-style: a per-batch bar with GPU memory, running loss
and both learning rates, then a per-class precision/recall/F1 table after each
epoch, then a confusion matrix at the end.

Run ids: `A0`–`A14`, and `B1_bert_frozen_cls`, `B2_bert_finetuned_cls`,
`B3_bert_frozen_mean_pool`, `B4_bert_scalar_mix`, `B5_token_attention_only`,
`B6_full_ldtf_frozen`, `B7_full_ldtf_finetuned`.

Add `--resume` to continue from `last.pt` if Colab disconnects.

In [ ]:
RUN  = 'A0'
SEED = 42

!python -m experiments.run_experiment --run $RUN --seed $SEED --batch-size 32


## 8. Train a suite with a live leaderboard

Presets: `core` (A0, A1, A3, A4, A11 — the comparisons that decide the
question), `baselines`, `ablations`, `all`.

The leaderboard redraws after every run, sorted by validation macro F1.
Completed runs are skipped on restart, so you can just rerun this cell after
a disconnect.

In [ ]:
!python -m experiments.run_suite --preset core --seed 42 --continue-on-error


### Watch the disk

Each run writes `best.pt` (~440 MB) and `last.pt` (~1.3 GB). Colab gives you
roughly 80 GB, so the full 19-configuration suite will not fit. `last.pt` is
only needed to resume, so delete it for runs that already finished.

In [ ]:
!df -h /content | tail -1

from pathlib import Path

DELETE_LAST = False   # True to reclaim space from completed runs

for summary in sorted(Path('outputs').glob('*/run_summary.json')):
    directory = summary.parent
    last = directory / 'last.pt'
    size = last.stat().st_size / 2**30 if last.exists() else 0.0
    if DELETE_LAST and last.exists():
        last.unlink()
        print(f'{directory.name:<32} freed {size:.2f} GB')
    else:
        print(f'{directory.name:<32} last.pt {size:.2f} GB')


### The classical baseline (B0)
TF-IDF + logistic regression. `C` is tuned on validation only.

In [ ]:
!python -m experiments.run_tfidf_baseline


## 9. Inspect what the routers learned

The scientific claim is that different classes attend to different layers.
If these rows are near-identical, LDTF has collapsed to a scalar mix and the
contribution is not real — regardless of accuracy.

In [ ]:
import torch
from src import config
from src.dataset import build_dataloaders
from src.evaluate import load_model_from_checkpoint
from src.models import LdtfBert
from src.train import forward_kwargs, move_batch_to_device
from src.utils import get_device

RUN_DIR = config.experiment_output_dir('A0_seed42')
model, _ = load_model_from_checkpoint(RUN_DIR / 'best.pt')
device = get_device(); model.to(device).eval()

tokenizer = LdtfBert.build_tokenizer(config.MODEL_NAME)
loaders = build_dataloaders(tokenizer, batch_size=64)
batch = move_batch_to_device(next(iter(loaders['validation'])), device)

with torch.no_grad():
    out = model(**forward_kwargs(batch), return_routing=True)
depth = out['depth_attention'].float().mean(0).cpu()   # [C, L]

print('Mean depth attention per class (rows sum to 1)\n')
print('class      ' + ''.join(f'L{i + 1:<5}' for i in range(depth.shape[1])))
for index, name in enumerate(config.LABEL_NAMES):
    print(f'{name:<11}' + ''.join(f'{v:<6.3f}' for v in depth[index].tolist()))

spread = (depth.max(0).values - depth.min(0).values).max().item()
print(f'\nlargest between-class gap on any layer: {spread:.4f}')
print('near zero would mean the classes agree, i.e. no label conditioning in practice')


In [ ]:
import matplotlib.pyplot as plt

figure, axis = plt.subplots(figsize=(9, 3.2))
image = axis.imshow(depth.numpy(), aspect='auto', cmap='viridis')
axis.set_yticks(range(len(config.LABEL_NAMES)))
axis.set_yticklabels(config.LABEL_NAMES)
axis.set_xticks(range(depth.shape[1]))
axis.set_xticklabels([f'L{i + 1}' for i in range(depth.shape[1])])
axis.set_title('Depth attention per class')
figure.colorbar(image)
plt.tight_layout()
plt.show()


## 10. Export the result tables

Rows that have not been run show `PENDING`. Nothing is estimated or back-filled.

In [ ]:
!python -m experiments.export_tables --with-params

from pathlib import Path
from IPython.display import Markdown, display

for name in ('baseline_table.md', 'ablation_table.md'):
    display(Markdown((Path('reports/tables') / name).read_text()))


## 11. Compare two runs properly

At n = 7,600 a single run's standard error is about 0.27 pp, so a gap under
roughly 0.75 pp is not distinguishable from noise. Use a paired test.

In [ ]:
import numpy as np
from src.metrics import bootstrap_accuracy_difference, mcnemar_exact

# Requires step 12 to have written test_predictions.npz for both runs.
A, B = 'A0_seed42', 'A1_seed42'
try:
    left  = np.load(config.experiment_output_dir(A) / 'test_predictions.npz')
    right = np.load(config.experiment_output_dir(B) / 'test_predictions.npz')
    labels = left['labels']
    test = mcnemar_exact(left['predictions'], right['predictions'], labels)
    ci = bootstrap_accuracy_difference(left['predictions'], right['predictions'], labels)
    print(f'{A} vs {B}')
    print(f"  McNemar b={test['b']:.0f} c={test['c']:.0f} p={test['p_value']:.4f}")
    print(f"  accuracy difference {ci['mean_difference']:+.4f} "
          f"95% CI [{ci['ci_lower_95']:+.4f}, {ci['ci_upper_95']:+.4f}]")
    if ci['ci_lower_95'] <= 0 <= ci['ci_upper_95']:
        print('  the interval crosses zero: report this as a null result')
except FileNotFoundError:
    print('Run step 12 first to produce test predictions.')


## 12. Locked final evaluation (run once, at the very end)

**Stop.** Only run this after every architecture, hyper-parameter, epoch and
seed decision is final. Using it earlier invalidates the results.

Access requires an explicit unlock token, is refused while training is active,
and appends a hash-stamped record to `reports/official_test_access.jsonl`. The
`access_index` in that ledger is how many times the test set has ever been
touched, and it belongs in the write-up.

In [ ]:
# Uncomment to unseal. Do this once.
#
# import os
# os.environ['LDTF_ALLOW_OFFICIAL_TEST'] = 'I_AM_REPORTING_FINAL_RESULTS'
# !python -m experiments.final_eval --run A0_seed42 --run A1_seed42 --reason 'final reported numbers'
print('sealed — uncomment above only when all model selection is complete')


## 13. Save results to Drive

Colab wipes `/content` when the runtime recycles. Copy `outputs/` and
`reports/` if you are not already working inside Drive.

In [ ]:
SAVE_TO_DRIVE = False

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    destination = '/content/drive/MyDrive/ldtf_bert_results'
    !mkdir -p $destination
    !cp -r outputs $destination/
    !cp -r reports $destination/
    print('saved to', destination)
else:
    print('set SAVE_TO_DRIVE = True to copy outputs/ and reports/ to Drive')
